# 01 — Basic Image Generation with Stable Diffusion

This notebook introduces text-to-image generation using the Hugging Face **Diffusers** library.
We'll cover:

1. Installing and loading a Stable Diffusion pipeline
2. Generating a single image from a text prompt
3. Controlling output with seeds, guidance scale, and inference steps
4. Saving outputs with the `src.image.pipeline` helpers

> **Research note:** This is a starting-point notebook.  All code cells marked `# TODO` are stubs waiting to be filled in.

**References:**
- Diffusers docs: https://huggingface.co/docs/diffusers
- Stable Diffusion paper: https://arxiv.org/abs/2112.10752

## 0. Environment setup

In [ ]:
# Install core dependencies (run once; skip if already installed)
# Recommended — uv (https://docs.astral.sh/uv/):
#   uv pip install -r ../requirements.txt
#
# Classic pip fallback:
#   pip install -r ../requirements.txt
#
# Uncomment ONE of the lines below to install inside this notebook:
# !uv pip install -r ../requirements.txt
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, '..')  # Make src/ importable from the notebook

from pathlib import Path
from IPython.display import display

OUTPUT_DIR = Path('../assets/outputs/01_image_gen')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the pipeline

We use the `ImageGenerationPipeline` wrapper from `src/image/pipeline.py`.
Under the hood it loads a Diffusers `StableDiffusionPipeline`.

In [ ]:
from src.image.pipeline import ImageGenerationPipeline

# TODO: Fill in src/image/pipeline.py before running this cell.
#       This stub will raise NotImplementedError until implemented.
MODEL_ID = 'runwayml/stable-diffusion-v1-5'
pipe = ImageGenerationPipeline(model_id=MODEL_ID, device='cuda', dtype=None)
print(f'Loaded pipeline: {MODEL_ID}')

## 2. Generate a single image

In [ ]:
PROMPT = 'a photorealistic portrait of an astronaut on the moon, dramatic lighting, 8k'
NEGATIVE_PROMPT = 'blurry, low quality, deformed'
SEED = 42

# TODO: Implement ImageGenerationPipeline.generate() in src/image/pipeline.py
image = pipe.generate(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED,
    num_inference_steps=30,
    guidance_scale=7.5,
)
display(image)
image.save(OUTPUT_DIR / 'basic_generation.png')
print('Saved to assets/outputs/01_image_gen/basic_generation.png')

## 3. Explore seed reproducibility

Re-running the cell below with the same seed should produce the **identical** image.
Change `SEED` to get a different result.

In [ ]:
from src.utils.seed_utils import lock_seed

for seed in [0, 42, 1337]:
    lock_seed(seed)  # Lock Python / NumPy / PyTorch RNGs
    img = pipe.generate(prompt=PROMPT, seed=seed, num_inference_steps=20)
    img.save(OUTPUT_DIR / f'seed_{seed}.png')
    display(img)

## 4. Guidance scale sweep

Higher guidance scale → image follows the prompt more closely but can look over-saturated.
Lower guidance scale → more creative / diverse outputs.

In [ ]:
for cfg_scale in [3.0, 7.5, 12.0, 20.0]:
    img = pipe.generate(
        prompt=PROMPT,
        seed=SEED,
        guidance_scale=cfg_scale,
        num_inference_steps=30,
    )
    img.save(OUTPUT_DIR / f'cfg_{cfg_scale:.0f}.png')
    print(f'guidance_scale={cfg_scale}')
    display(img)

## 5. Batch generation with `generate_batch`

In [ ]:
prompts = [
    'a red fox in a snowy forest, photorealistic',
    'a golden retriever on a beach at sunset',
    'a black cat sitting on a rooftop under the stars',
]

images = pipe.generate_batch(prompts, seed=42)
saved_paths = pipe.save_images(images, OUTPUT_DIR / 'batch', prefix='animal')
for img in images:
    display(img)

---
### Next steps
- **Notebook 02**: LoRA / IP-Adapter / DreamBooth consistency techniques
- **Notebook 03**: Prompt consistency experiments and CLIP scoring